# ConceptGraphs baseline with local Qwen / SmolVLM

This notebook runs the **original ConceptGraphs detector, 3D mapper and scene graph builder**.
Qwen and SmolVLM replace LLaVA/GPT for object naming and relationship answers. No API key is needed.

```text
OSMa RGB-D + camera calibration + traj.txt
  -> generate_gsa_results (RAM / GroundingDINO / SAM / CLIP)
  -> cfslam_pipeline_batch (3D object fusion)
  -> local Qwen / SmolVLM object labels
  -> build_scenegraph_cfslam.build_scenegraph
       original pruning + overlap + minimum spanning forest
       local Qwen / SmolVLM relationship answers
  -> graph.json + original upstream outputs + review.csv
```

**Scene graph construction runs in section 5**, through `conceptgraphs_support.conceptgraphs_local_graph.run_local_graph()`.
That function calls upstream `build_scenegraph()` and `generate_scenegraph_json()`.
This is a local-model ConceptGraphs baseline; it is not a reproduction of the paper's LLaVA/GPT scores.
The local adapters use image crops and marked image pairs, matching their training input format.

Use your existing **cleanjupyter** Windows kernel for the VLMs. The older mapping dependencies run in an
isolated **local Ubuntu WSL** environment on this same computer. Training remains local.
The setup helper downloads dependency files; inference itself is offline.

In [ ]:
from pathlib import Path
import copy, importlib, json, os, subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "code/midterm_report/lora_baseline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside your Senior Capstone repository.")
sys.path.insert(0, str(ROOT / "code/midterm_report"))
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
import lora_baseline as lb
from conceptgraphs_support import conceptgraphs_bridge as cg
from conceptgraphs_support import scene_graph_pipeline as sg
for module in (lb, cg, sg):
    importlib.reload(module)
print("VLM worker / notebook Python:", sys.executable)
print("Project:", ROOT)

## 1. Action and pilot data

| Action | Work performed |
|---|---|
| `inspect` | Check local checkpoint paths, selected scenes and mapping imports. |
| `setup` | Download and install the separate local WSL mapping runtime. Resumes cached downloads. |
| `adapter_smoke` | Small real VLM component check using saved validation examples. Does not build a graph. |
| `prepare` | Stage the selected RGB-D frames and poses. |
| `map` | Stage, detect and fuse the original ConceptGraphs 3D object map. |
| `graph` | Build actual scene graphs from that saved map with the selected local VLMs. |
| `all` | Stage, detect, map and build scene graphs. |

First use `inspect`. If the runtime is missing, run `setup`; then set `ACTION="all"` and Run All.
To repeat only graph construction, keep the same experiment and set `ACTION="graph"`.
Use a new experiment name after changing data, detector or mapping settings. Graph reruns get new folders.
Sampling caps can be `None`; frame end is exclusive. The pilot defaults are editable.

In [ ]:
ACTION = "inspect"
EXPERIMENT_NAME = "pilot_local_01"
DATA_ROOT = ROOT / "dataset/osma-bench/data"
OUTPUT_ROOT = ROOT / "artifacts/scene_graphs"
CONCEPTGRAPHS_ROOT = ROOT / "dependencies/concept-graphs"
RUNTIME_ROOT = ROOT / "dependencies/conceptgraphs-runtime"
SOURCE_MANIFEST = ROOT / "artifacts/objects/qwen/objects_3epochs/data_manifest.json"

WSL_DISTRO = "Ubuntu"  # Set None to use a compatible native mapping Python instead.
MAPPING_PYTHON = sys.executable  # Used only when WSL_DISTRO is None.
LINUX_PYTHON = None  # None discovers the isolated environment below; or set an exact path.
SPLIT = "validation"
SCENES = None          # Or an explicit list of scene names in this split.
MAX_SCENES = 1         # None uses every scene in the selected split.
FRAME_SELECTION = dict(frame_start=0, frame_end=None, frame_stride=10, max_frames=30, frame_ids=None)
CAMERA = dict(pose_convention="camera_to_world", camera_axes="opencv", camera_overrides={},
              image_height=None, image_width=None)  # None reads each scene's camera YAML.
DEVICE = "cuda"
SEED = 42
FILE_MODE = "copy"     # Or hardlink for same-volume storage.
SMOKE_EXAMPLES_PER_TASK = 1

if ACTION not in {"inspect", "setup", "adapter_smoke", "prepare", "map", "graph", "all"}:
    raise ValueError("Choose an ACTION from the table.")
if not EXPERIMENT_NAME or Path(EXPERIMENT_NAME).name != EXPERIMENT_NAME or EXPERIMENT_NAME in {".", ".."}:
    raise ValueError("EXPERIMENT_NAME must be a single folder name.")
EXPERIMENT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
selected_scenes = sg.select_scenes(DATA_ROOT, SOURCE_MANIFEST, SPLIT, SCENES, MAX_SCENES)
display(pd.DataFrame(selected_scenes))
display(pd.DataFrame(sg.local_runs(ROOT)))

## 2. One-time local mapping runtime

`setup` uses Windows networking to cache the older Linux packages, source archives and detector weights,
then installs them offline into `~/miniconda3/envs/conceptgraphs-pilot` in Ubuntu WSL.
It does not change the Windows training environment. Initial downloads are several GB; a network failure
stops setup with an error. Completed files and partial downloads remain available for retry.

You can use your own compatible environment and checkpoint paths instead. The install helper's separate
stages are `resolve`, `conda`, `sources`, `torch`, `pip`, `weights`, `pytorch3d`, and `install`.
`--env-prefix`, `--linux-conda`, `--runtime-root`, `--build-jobs`, and `--cuda-arch` are configurable.
The default CUDA architecture 8.9 matches this machine's RTX 4060.

In [ ]:
# Windows Anaconda base Python is used only for downloading / setup.
SETUP_PYTHON = next((p / "python.exe" for p in (Path(sys.prefix), *Path(sys.prefix).parents)
                     if (p / "Scripts/conda.exe").is_file()), None)
SETUP_EXTRA_ARGS = []  # Example: ["--build-jobs", "2", "--cuda-arch", "8.9"]
if ACTION == "setup":
    if WSL_DISTRO is None or SETUP_PYTHON is None:
        raise RuntimeError("This setup helper requires Windows Anaconda and a local WSL distro.")
    subprocess.run([str(SETUP_PYTHON), "-u", str(ROOT / "code/setup/setup_conceptgraphs_runtime.py"),
                    "all", "--distro", WSL_DISTRO, "--runtime-root", str(RUNTIME_ROOT), *SETUP_EXTRA_ARGS], check=True)

runtime_available = True
if WSL_DISTRO is not None:
    if LINUX_PYTHON is None:
        installed = RUNTIME_ROOT / "installed.json"
        if installed.is_file():
            installed_settings = json.loads(installed.read_text())
            LINUX_PYTHON = installed_settings["python"]
        else:
            linux_home = subprocess.check_output(["wsl.exe", "-d", WSL_DISTRO, "--", "printenv", "HOME"], text=True).strip()
            LINUX_PYTHON = linux_home + "/miniconda3/envs/conceptgraphs-pilot/bin/python"
    runtime_available = subprocess.run(["wsl.exe", "-d", WSL_DISTRO, "--", "test", "-x", LINUX_PYTHON]).returncode == 0
print("Mapping Python:", LINUX_PYTHON if WSL_DISTRO else MAPPING_PYTHON)
print("Mapping interpreter exists:", runtime_available)

## 3. Detector and mapping parameters

These are concrete local paths populated by `setup`. Qwen/SmolVLM adapters are separate from the
RAM, GroundingDINO, SAM and CLIP weights used to build the object map.
`MAPPING` accepts upstream Hydra algorithm settings. Change dataset/frame settings in section 1.
No semantic masks, object class dictionaries or VQA answers are used to construct the graph.
Calibration and depth scale are read separately for each HM3D / ReplicaCAD scene.
CLIP runs on CPU by default to leave GPU memory for SAM, DINO and RAM; `clip_device` is editable.

In [ ]:
GSA = RUNTIME_ROOT / "src/Grounded-Segment-Anything"
WEIGHTS = RUNTIME_ROOT / "weights"
DETECTOR = copy.deepcopy(cg.DETECTOR_DEFAULTS)
DETECTOR.update(class_set="ram", detector="dino", sam_variant="sam",
                box_threshold=0.25, text_threshold=0.25, nms_threshold=0.5)
DETECTOR_RUNTIME = copy.deepcopy(cg.RUNTIME_DEFAULTS)
DETECTOR_RUNTIME.update(
    gsa_path=str(GSA),
    dino_config=str(GSA / "GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py"),
    dino_checkpoint=str(WEIGHTS / "groundingdino_swint_ogc.pth"),
    dino_text_encoder_path=str(WEIGHTS / "bert-base-uncased"),
    sam_checkpoint=str(WEIGHTS / "sam_vit_h_4b8939.pth"), sam_encoder_version="vit_h",
    ram_checkpoint=str(WEIGHTS / "ram_swin_large_14m.pth"),
    clip_model="ViT-H-14", clip_checkpoint=str(WEIGHTS / "clip_vit_h14.bin"),
    clip_crop_padding=20, clip_device="cpu",  # Keep ViT-H off the 8 GB GPU; change to cuda if desired.
)
MAPPING = copy.deepcopy(cg.MAPPING_DEFAULTS)
MAPPING.update(
    spatial_sim_type="overlap", match_method="sim_sum", sim_threshold=1.2,
    downsample_voxel_size=0.025, dbscan_eps=0.1, dbscan_min_points=10,
    obj_min_points=20, obj_min_detections=3, mask_conf_threshold=0.25,
    merge_overlap_thresh=0.7, merge_visual_sim_thresh=0.8, merge_text_sim_thresh=0.8,
)

## 4. Qwen / SmolVLM conditions

The four default conditions include pretrained and adapted models. Both have saved object adapters;
only Qwen currently has a saved relationship adapter. SmolVLM's condition explicitly records its
pretrained relationship stage. To use a future SmolVLM relationship adapter, set its `relationship_run`
and `relationship_adapter=True`. Requested missing adapters raise an error.

Models load one task at a time in a separate local worker. All conditions start from the same 3D map,
but local naming can affect the original invalid-node pruning, so retained nodes and MST pairs can differ.
Votes retain ties as unresolved. Pairs without a shared RGB view remain unresolved, not predicted `none`.
`next_to` and `nearby` extend the original relation vocabulary to match the trained adapters.

The upstream builder retains its original overlap cutoff 0.01 and 25-second per-pair timeout;
timeouts are recorded as failures. These are upstream algorithm choices, not training limits.

In [ ]:
VARIANTS = sg.default_variants(ROOT)
# For a faster initial run: VARIANTS = [v for v in VARIANTS if v["name"] == "qwen_adapted"]
LOADING = dict(device="cuda:0", use_4bit=True, dtype="auto", attn_implementation="sdpa")
# Optional loading overrides: qwen_min_pixels, qwen_max_pixels, smolvlm_image_splitting,
# model_path, model_kwargs. Saved processor settings are otherwise preserved.
GENERATION = dict(max_new_tokens=24, do_sample=False, seed=SEED)
GRAPH = dict(
    worker_python=sys.executable, variants=VARIANTS, split=SPLIT,
    loading=LOADING, generation=GENERATION,
    object_views=3, pair_views=2,       # None uses all available views.
    min_views_per_object=2,
    pair_point_distance_m=None,        # None uses mapping's voxel size.
    object_image_max_side=384, relation_image_max_side=512,
    object_prompt=lb.PROMPT, relation_prompt=lb.RELATION_PROMPT,
)
for variant in VARIANTS:
    sg.validate_variant(ROOT, variant)
display(pd.DataFrame(VARIANTS))

bridge_configs = {}
for selection in selected_scenes:
    scene = selection["scene"]
    bridge_configs[scene] = cg.BridgeConfig(
        data_root=str(DATA_ROOT), scene=scene,
        run_dir=str(EXPERIMENT_DIR / scene.replace("/", "__") / "mapping"),
        conceptgraphs_root=str(CONCEPTGRAPHS_ROOT), python_executable=str(MAPPING_PYTHON),
        device=DEVICE, seed=SEED, file_mode=FILE_MODE, **FRAME_SELECTION, **CAMERA,
        detector=DETECTOR, detector_runtime=DETECTOR_RUNTIME, mapping=MAPPING, local_graph=GRAPH,
    )
if ACTION == "inspect":
    # Check Windows-visible files even before the Linux interpreter is installed.
    report = cg.preflight(next(iter(bridge_configs.values())), check_imports=WSL_DISTRO is None)
    display(pd.DataFrame(report["checks"]))
    if WSL_DISTRO and runtime_available:
        cg.run_local_pipeline(next(iter(bridge_configs.values())), ("inspect",),
                              wsl_distro=WSL_DISTRO, linux_python=LINUX_PYTHON)
    elif not runtime_available:
        print("Mapping runtime missing. Run ACTION='setup' before ACTION='all'.")

In [ ]:
if ACTION == "adapter_smoke":
    smoke_dir = lb.available_run_directory(OUTPUT_ROOT / "adapter_smoke")
    smoke_dir.mkdir(parents=True)
    smoke = sg.adapter_smoke(ROOT, DATA_ROOT, VARIANTS, samples=SMOKE_EXAMPLES_PER_TASK,
                             loading=LOADING, generation=GENERATION)
    lb.save_json(smoke_dir / "results.json", smoke)
    display(pd.DataFrame(smoke["results"]))
    print("Component check only:", smoke_dir)

## 5. Run the actual ConceptGraphs pipeline

This cell runs detection and mapping in separate local processes, followed by the **actual upstream
scene graph builder**. Its GPT interface is replaced in-process with a local Qwen/SmolVLM worker;
there is no remote fallback. The original source checkout is left unchanged.

`map` stops at the 3D map. **`graph` runs graph construction; `all` includes it.**
Completed detection/map stages are reused only when the configuration and source hashes match.
Every graph run gets a fresh output folder. Errors propagate; an incomplete run is never marked complete.

In [ ]:
STAGES = {
    "prepare": ("prepare",),
    "map": ("prepare", "detect", "map"),
    "graph": ("graph",),
    "all": ("prepare", "detect", "map", "graph"),
}
if ACTION in STAGES:
    if not runtime_available:
        raise RuntimeError("The local mapping interpreter is missing. Run ACTION='setup' first.")
    for scene, config in bridge_configs.items():
        print("Running", scene, STAGES[ACTION], flush=True)
        cg.run_local_pipeline(config, STAGES[ACTION], wsl_distro=WSL_DISTRO, linux_python=LINUX_PYTHON)
else:
    print("Pipeline skipped for ACTION=", ACTION)

## 6. Read graphs and inspect problems

Outputs live under `artifacts/scene_graphs/<experiment>/<scene>/scene_graphs/<variant>/`:

- `graph.json`: nodes, valid edges and unresolved pairs, with original map IDs preserved.
- `local_node_predictions.json` / `local_relation_predictions.json`: crops, views, raw answers and votes.
- `review.csv`: candidate-pair decisions with blank review labels and failure-stage columns.
- `run.json`: exact local settings and completion status; `*_worker.log`: model loading/errors.
- `cfslam_scenegraph_edges.pkl`, `map/scene_map_cfslam_pruned.pkl.gz` and `scene_graph.json`: upstream outputs.

Some original output names contain `gpt-4`; this is only a filename compatibility requirement.
They contain local model results. No GPT calls are made.

Review failures by stage: detection, fusion, node naming/pruning, candidate selection, relationship
prediction or insufficient shared-view evidence. A sparse graph is not automatically a VLM failure:
the minimum spanning forest selects few pairs before the relationship model sees anything.
Accuracy needs independently reviewed nodes and pairs, including positive pairs the selector missed.
The existing relationship data are imbalanced; raw graph counts are diagnostics, not accuracy scores.

In [ ]:
comparison_rows = []
for scene, config in bridge_configs.items():
    comparison_path = Path(config.run_dir).parent / "scene_graphs/comparison.json"
    if comparison_path.is_file():
        for row in json.loads(comparison_path.read_text()):
            # Runtime paths may be Linux paths; resolve the folder by its name on Windows.
            graph_dir = comparison_path.parent / row["output"].replace("\\", "/").rstrip("/").split("/")[-1]
            comparison_rows.append({"scene": scene, **row, "output": str(graph_dir)})
            graph = json.loads((graph_dir / "graph.json").read_text())
            print(scene, row["variant"], "->", graph_dir)
            display(pd.DataFrame(graph["nodes"]).head(10))
            display(pd.DataFrame(graph["edges"]).head(15))
if comparison_rows:
    display(pd.DataFrame(comparison_rows))
else:
    print("No completed scene graph yet for this experiment.")

## Further local training

Use [object training](finetune_vlm.ipynb) and [relationship training](finetune_vlm2.ipynb).
Their data, loading, LoRA, TrainingArguments and generation dictionaries expose epochs, learning rate,
batches, accumulation, scheduler, precision, checkpointing, image budgets and sampling limits.
Use the newly saved run paths in `VARIANTS` when comparing them. Preserve held-out scene/layout splits.

The earlier fixed-map comparison helpers remain in `conceptgraphs_support/scene_graph_pipeline.py` for ablations;
this notebook's `graph` action uses the original ConceptGraphs graph-construction function.